# J13 — N01 model XAI: original → exact prepared → Grad-CAM / Grad-CAM++ / Score-CAM

## 1. Input selection, checkpoint provenance

In [1]:
import os,io,json,hashlib,time,zipfile,subprocess,gc
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from torch.nn import functional as F
from torchvision import transforms
from torchvision.models import resnet50
from PIL import Image,ImageOps
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC'];SRC=['rotation','simclr','FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion']
INPUT=Path('/kaggle/input' if Path('/kaggle/input').exists() else '/mnt/data')
WORK=Path('/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data')
OUT=WORK/'j13-lesion-aware-gradcam-plus-scorecam'
for k in ['figures','reports','maps','cache']:(OUT/k).mkdir(parents=True,exist_ok=True)
TEMP=Path('/kaggle/temp/j13' if Path('/kaggle/temp').exists() else '/mnt/data/j13_tmp');TEMP.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type!='cuda':raise RuntimeError('Enable Kaggle GPU for Grad-CAM++ and Score-CAM')
SEED=42;START=time.monotonic();HOURS=float(os.getenv('J13_MAX_HOURS','8'))
MAX_PER_SOURCE=int(os.getenv('J13_PER_SOURCE','3'))
SCORECAM_CHANNELS=int(os.getenv('J13_SCORECAM_CHANNELS','24'))
SCORECAM_BATCH=int(os.getenv('J13_SCORECAM_BATCH','8'))
NORM=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
MEAN=torch.tensor([.485,.456,.406],device=DEVICE).reshape(1,3,1,1)
STD=torch.tensor([.229,.224,.225],device=DEVICE).reshape(1,3,1,1)
def req(v,msg):
    if not bool(v):raise RuntimeError(msg)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def write(p,o):Path(p).write_text(json.dumps(o,indent=2,allow_nan=False,default=str))
def findstage(tag):
    matches=[]
    fn='run_summary.json' if tag=='R04' else 'final_status.json'
    for p in INPUT.rglob(fn):
        try:
            d=json.loads(p.read_text())
            if d.get('stage')==tag and d.get('status')=='COMPLETED':matches.append((p.parent.parent,d))
        except (OSError,ValueError):continue
    req(len(matches)==1,f'Attach exactly one completed {tag} notebook output; found {len(matches)}')
    return matches[0]
R04,run04=findstage('R04');R06,run06=findstage('R06')
req(run04['n01_manifest_sha256']==run06['n01_manifest_sha256'],'R04/R06 source manifest mismatch')
# Candidate-level validation probabilities are used only to audit exact checkpoint/image reproduction.
paths=[]
for p in INPUT.rglob('final_status.json'):
    try:
        d=json.loads(p.read_text())
        if d.get('stage')=='G03-B' and d.get('status')=='COMPLETED':paths.append(p.parent.parent/'predictions/ensemble_probabilities.npz')
    except Exception:continue
req(len(paths)==1 and paths[0].is_file(),'Attach completed G03-B output')
with np.load(paths[0],allow_pickle=True) as z:
    names=z['candidate_names'].astype(str).tolist();ids=z['image_id'].astype(str);y=z['y'].astype(int);origin=z['source'].astype(str)
    req('Base_rotation' in names and 'Base_simclr' in names,'Expected R04 base candidates missing')
    ref={s:np.asarray(z['proba'][names.index('Base_'+s)],dtype=np.float32) for s in ['rotation','simclr']}
req(len(set(ids))==len(ids),'Duplicate validation image identities')
df=pd.DataFrame({'image_id':ids,'y':y,'source':origin})
df['hash']=df.image_id.map(lambda v:hashlib.sha256(('XAI42/'+v).encode()).hexdigest())
selected=[]
for source in ['HAM10000','BCN20000']:
    d=df[df.source.eq(source)].sort_values('hash')
    # Deterministic class-diverse selection, not based on confidence or success.
    chosen=d.drop_duplicates('y').head(MAX_PER_SOURCE)
    if len(chosen)<MAX_PER_SOURCE:
        chosen=pd.concat([chosen,d.loc[~d.image_id.isin(chosen.image_id)].head(MAX_PER_SOURCE-len(chosen))])
    selected.append(chosen)
chosen=pd.concat(selected).reset_index(drop=True)
chosen[['image_id','source','y']].to_csv(OUT/'reports/selected_images.csv',index=False)
print('XAI examples:',len(chosen),chosen[['image_id','source','y']].to_string(index=False))


XAI examples: 6     image_id   source  y
ISIC_0028428 HAM10000  1
ISIC_0031138 HAM10000  4
ISIC_0027155 HAM10000  0
ISIC_0066252 BCN20000  1
ISIC_0066359 BCN20000  0
ISIC_0064452 BCN20000  2


## 2. Exact R04/R06 models with differentiable fusion

In [2]:
DROPOUT=.35;SOURCES=SRC
class R04Classifier(nn.Module):
    def __init__(self,src):
        super().__init__()
        self.encoder=resnet50(weights=None);self.encoder.fc=nn.Identity()
        self.classifier=nn.Linear(2048,8)
        best=torch.load(R04/'models'/src/'best.pt',map_location='cpu',weights_only=False)
        enc=torch.load(R04/'models'/src/'encoder_best.pt',map_location='cpu',weights_only=False)
        req(best['n01_manifest_sha256']==run04['n01_manifest_sha256'],'Wrong R04 model provenance')
        req(enc['n01_manifest_sha256']==run04['n01_manifest_sha256'],'Wrong encoder provenance')
        req(enc.get('stage')=='R04' and enc.get('source')==src and enc.get('feature_dim')==2048 and enc.get('fc')=='Identity','Wrong R04 exported encoder')
        req(enc.get('signature')==best.get('signature'),'R04 encoder/model signature mismatch')
        req(all(torch.equal(v,best['model']['encoder.'+k]) for k,v in enc['encoder_state'].items()),'R04 exported encoder differs from selected classifier')
        self.encoder.load_state_dict(enc['encoder_state'],strict=True)
        self.classifier.load_state_dict({'weight':best['model']['classifier.weight'],'bias':best['model']['classifier.bias']},strict=True)
    def forward(self,x):
        f=self.encoder(x);return f,self.classifier(f)
encoders={s:R04Classifier(s).to(DEVICE).eval() for s in ['rotation','simclr']}
normalization=R06/'reports/train_only_normalization.npz';req(normalization.is_file(),'Missing R06 feature normalization')
with np.load(normalization,allow_pickle=False) as z:
    stats={s:(np.asarray(z[s+'_mean'],dtype=np.float32),np.asarray(z[s+'_std'],dtype=np.float32)) for s in ['rotation','simclr']}
for s,(m,sd) in stats.items():req(m.shape==(2048,) and sd.shape==(2048,) and np.all(sd>0),'Invalid training-only normalization')
from torch.nn import functional as F
class FusionConcat(nn.Module):
    def __init__(self):
        super().__init__();self.body=nn.Sequential(nn.LayerNorm(4096),nn.Linear(4096,512),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(512,128),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(128,8))
    def forward(self,r,s):return self.body(torch.cat([r,s],dim=1))
class FusionGatedV2(nn.Module):
    def __init__(self):
        super().__init__();self.r=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256),nn.GELU())
        self.s=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256),nn.GELU())
        self.gate=nn.Sequential(nn.Linear(512,256),nn.Sigmoid())
        self.classify=nn.Sequential(nn.LayerNorm(256),nn.Dropout(DROPOUT),nn.Linear(256,8))
    def forward(self,r,s):
        a=self.r(r);b=self.s(s);g=self.gate(torch.cat([a,b],1))
        return self.classify(g*a+(1-g)*b)
class FusionGatedV3(nn.Module):
    def __init__(self):
        super().__init__();self.r=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256),nn.GELU())
        self.s=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256),nn.GELU())
        self.gate=nn.Linear(512,512)
        self.classify=nn.Sequential(nn.LayerNorm(512),nn.Linear(512,128),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(128,8))
    def forward(self,r,s):
        a=self.r(r);b=self.s(s);g=torch.softmax(self.gate(torch.cat([a,b],1)).reshape(-1,2,256),dim=1)
        mix=g[:,0,:]*a+g[:,1,:]*b
        return self.classify(torch.cat([mix,torch.abs(a-b)],1))
class ResidualFusion(nn.Module):
    def __init__(self):
        super().__init__();self.r=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256))
        self.s=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256))
        self.res=nn.Sequential(nn.LayerNorm(512),nn.Linear(512,256),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(256,256))
        self.classify=nn.Sequential(nn.LayerNorm(256),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(256,8))
    def forward(self,r,s):
        a=self.r(r);b=self.s(s);return self.classify(.5*(a+b)+self.res(torch.cat([a,b],1)))
MODELS={'FusionConcat':FusionConcat,'FusionGatedV2':FusionGatedV2,'FusionGatedV3':FusionGatedV3,'ResidualFusion':ResidualFusion}
fusions={}
for s in SOURCES[2:]:
    p=R06/'models'/f'{s}_best.pt';req(p.is_file(),f'Missing R06 checkpoint {p}')
    ck=torch.load(p,map_location='cpu',weights_only=False)
    cfg=ck['config'];req(cfg['classes']==CLASSES and cfg['name']==s and cfg['fit_partition']=='train','Wrong trained R06 configuration')
    req(cfg['r05_manifest_sha256']==run04['n01_manifest_sha256'],'R06 checkpoint from different N01 manifest')
    m=MODELS[s]().to(DEVICE).eval();m.load_state_dict(ck['model'],strict=True);fusions[s]=m
print('Fusion checkpoints verified:',list(fusions))
class ImageToLogits(nn.Module):
    def __init__(self,name):
        super().__init__();self.name=name
        if name in ('rotation','simclr'):self.base=encoders[name]
        else:
            self.r=encoders['rotation'];self.s=encoders['simclr'];self.head=fusions[name]
        self.register_buffer('rmean',torch.as_tensor(stats['rotation'][0]).view(1,-1))
        self.register_buffer('rstd',torch.as_tensor(stats['rotation'][1]).view(1,-1))
        self.register_buffer('smean',torch.as_tensor(stats['simclr'][0]).view(1,-1))
        self.register_buffer('sstd',torch.as_tensor(stats['simclr'][1]).view(1,-1))
    def forward(self,x):
        if self.name in ('rotation','simclr'):
            _,z=self.base(x);return z
        r=self.r.encoder(x);s=self.s.encoder(x)
        rn=torch.clamp((r-self.rmean)/self.rstd,-12,12)
        sn=torch.clamp((s-self.smean)/self.sstd,-12,12)
        return self.head(rn,sn)
    def target_layer(self,branch='simclr'):
        source=self.name if self.name in ('rotation','simclr') else branch
        owner=encoders[source].encoder
        return owner.layer4[-1]
wrappers={s:ImageToLogits(s).to(DEVICE).eval() for s in SRC}
counts={}
for name in SRC:
    models=[encoders[name]] if name in ('rotation','simclr') else [encoders['rotation'],encoders['simclr'],fusions[name]]
    counts[name]=sum(sum(t.numel() for t in m.parameters()) for m in models)/1e6
write(OUT/'reports/trained_parameter_counts.json',{f'lesion_aware_{k}':v for k,v in counts.items()})
print('Models loaded; trainable parameter totals (M):',counts)


Fusion checkpoints verified: ['FusionConcat', 'FusionGatedV2', 'FusionGatedV3', 'ResidualFusion']
Models loaded; trainable parameter totals (M): {'rotation': 23.524424, 'simclr': 23.524424, 'FusionConcat': 49.2214, 'FusionGatedV2': 48.240024, 'FusionGatedV3': 48.436504, 'ResidualFusion': 48.30684}


## 3. Original and preprocessed image pairs; R04 equivalence audit

In [3]:
# Exact N01 image pipeline from N01 3.1.0; original raw image is required for the first panel.
def n01_jpeg(source):
    with Image.open(io.BytesIO(source)) as img:
        img.load();rgb=ImageOps.exif_transpose(img).convert('RGB')
    t=np.asarray(rgb.resize((32,32),Image.Resampling.BILINEAR))
    edges=np.concatenate([t[0],t[-1],t[:,0],t[:,-1]],axis=0)
    color=tuple(np.median(edges,axis=0).astype(np.uint8).tolist())
    processed=ImageOps.pad(rgb,(224,224),method=Image.Resampling.BICUBIC,color=color)
    b=io.BytesIO();processed.save(b,format='JPEG',quality=94,subsampling=0,optimize=True)
    return rgb,b.getvalue()
archives=[p for p in INPUT.rglob('ISIC_2019_Training_Input.zip') if p.is_file()]
req(len(archives)<=1,'Multiple official source ZIPs; attach only one')
archive=archives[0] if archives else TEMP/'ISIC_2019_Training_Input.zip'
if not archive.is_file():
    print('Downloading official ISIC2019 source images...')
    run=subprocess.run(['curl','-fL','--retry','8','--connect-timeout','40','-o',str(archive),
      'https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip'],check=False)
    req(run.returncode==0 and archive.stat().st_size>5*10**9,'Official ISIC ZIP download failed; enable Internet')
with zipfile.ZipFile(archive) as z:
    members={}
    for name in z.namelist():
        if name.lower().endswith(('.jpg','.jpeg','.png')):
            ident=Path(name).stem;req(ident not in members,'Duplicate source ZIP image IDs')
            members[ident]=name
req(set(chosen.image_id).issubset(members),'Some selected images absent from official ZIP')
# Prefer exact N01 images when N01 source is attached; otherwise use reprocessed bytes.
prepared={}
for p in sorted(INPUT.rglob('experiment_master_manifest.csv')):
    root=p.parent.parent
    if not (root/'images').is_dir():continue
    meta=pd.read_csv(p,dtype=str,keep_default_na=False)
    if not {'image_id','image_relpath'}.issubset(meta.columns):continue
    meta=meta.drop_duplicates('image_id').set_index('image_id')
    paths={}
    for ident in chosen.image_id:
        if ident not in meta.index:break
        rel=Path(meta.loc[ident,'image_relpath']);req(not rel.is_absolute() and '..' not in rel.parts,'Unsafe N01 image path')
        q=root/rel
        if not q.is_file():break
        paths[ident]=q
    if len(paths)==len(chosen):req(not prepared,'Multiple complete N01 prepared image inputs');prepared=paths

def get_images(ident):
    with zipfile.ZipFile(archive) as z:data=z.read(members[ident])
    raw,regenerated=n01_jpeg(data)
    pix=prepared[ident].read_bytes() if prepared else regenerated
    with Image.open(io.BytesIO(pix)) as img:
        img.load();processed=img.convert('RGB')
    x=NORM(processed).unsqueeze(0).to(DEVICE)
    return raw,processed,x
# Sanity check against original R04 validation probabilities before generating explanations.
audit=[]
for r in chosen.itertuples():
    raw,processed,x=get_images(r.image_id)
    i=int(np.where(ids==r.image_id)[0][0])
    with torch.no_grad():
        for s in ['rotation','simclr']:
            p=torch.softmax(wrappers[s](x),1).cpu().numpy()[0]
            audit.append({'image_id':r.image_id,'model':s,'max_abs_error':float(np.abs(p-ref[s][i]).max())})
pd.DataFrame(audit).to_csv(OUT/'reports/r04_probability_equivalence.csv',index=False)
worst=max(a['max_abs_error'] for a in audit)
req(worst<.025,f'R04/feature-input mismatch {worst:.4f}. Attach exact N01 images or inspect preprocessing; no XAI generated.')
print('R04 reproduction audit passed, worst:',worst,'image mode:', 'N01 prepared' if prepared else 'reconstructed N01')


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9318M  100 9318M    0     0  27.5M      0  0:05:38  0:05:38 --:--:-- 16.4M


R04 reproduction audit passed, worst: 5.960464477539062e-07 image mode: reconstructed N01


## 4. Grad-CAM, Grad-CAM++, Score-CAM

In [4]:
def normmap(cam,size=(224,224)):
    cam=F.interpolate(cam[:,None],size=size,mode='bilinear',align_corners=False)[:,0]
    cam=torch.nan_to_num(cam,0.).clamp(min=0)
    mn=cam.flatten(1).amin(1).reshape(-1,1,1);mx=cam.flatten(1).amax(1).reshape(-1,1,1)
    return ((cam-mn)/(mx-mn).clamp_min(1e-10)).detach().cpu().numpy()[0]

def gradients_map(model,x,method,branch='simclr'):
    acts=[];hook=model.target_layer(branch).register_forward_hook(lambda mod,args,out:acts.append(out))
    try:
        with torch.enable_grad():
            z=model(x.detach().requires_grad_(True));c=int(z.detach().argmax(1).item())
            req(len(acts)==1,'Expected exactly one target encoder feature map')
            g=torch.autograd.grad(z[0,c],acts[0],retain_graph=False,create_graph=False)[0]
            a=acts[0].detach()
            if method=='gradcam':weight=g.mean((2,3),keepdim=True)
            elif method=='gradcam++':
                g2=g.square();g3=g2*g
                alpha=g2/(2*g2+(a*g3).sum((2,3),keepdim=True)+1e-8)
                weight=(alpha*F.relu(g)).sum((2,3),keepdim=True)
            else:raise RuntimeError('Unsupported gradient method')
            cam=F.relu((weight*a).sum(1))
        return normmap(cam),c,float(torch.softmax(z.detach(),1)[0,c])
    finally:hook.remove()

@torch.no_grad()
def scorecam(model,x,c,branch='simclr',max_channels=SCORECAM_CHANNELS):
    feats=[];hook=model.target_layer(branch).register_forward_hook(lambda m,inp,out:feats.append(out.detach()))
    try:
        _=model(x)
    finally:hook.remove()
    req(len(feats)==1,'ScoreCAM features were not captured')
    a=feats[0].detach()[0];channel_energy=a.flatten(1).square().mean(1)
    cids=torch.topk(channel_energy,k=min(max_channels,len(channel_energy))).indices
    maps=F.interpolate(a[cids,None],size=x.shape[-2:],mode='bilinear',align_corners=False)[:,0]
    low=maps.flatten(1).amin(1).view(-1,1,1);high=maps.flatten(1).amax(1).view(-1,1,1)
    maps=((maps-low)/(high-low).clamp_min(1e-8)).clamp(0,1)
    rgb=(x*STD+MEAN).clamp(0,1)
    vals=[]
    for start in range(0,len(cids),SCORECAM_BATCH):
        mask=maps[start:start+SCORECAM_BATCH,None];masked=((rgb*mask)-MEAN)/STD
        vals.append(torch.softmax(model(masked),1)[:,c])
    values=torch.cat(vals);weights=torch.softmax(values,dim=0)
    cam=(weights[:,None,None]*maps).sum(0,keepdim=True)
    return normmap(cam),len(cids)


## 5. High-resolution multi-model XAI panels

In [5]:
def panel(raw,processed,heatmaps,annotations,dest):
    names=SRC;fig,axs=plt.subplots(len(names),5,figsize=(17,3.2*len(names)),squeeze=False)
    for i,name in enumerate(names):
        panels=[np.asarray(raw),np.asarray(processed)]
        for j,raw_img in enumerate(panels):
            axs[i,j].imshow(raw_img);axs[i,j].axis('off');axs[i,j].set_title(('Original' if j==0 else 'N01 preprocessed') if i==0 else '')
        for j,kind in enumerate(['gradcam','gradcam++','scorecam'],start=2):
            axs[i,j].imshow(processed);axs[i,j].imshow(heatmaps[name][kind],cmap='turbo',alpha=.43,vmin=0,vmax=1)
            axs[i,j].axis('off')
            if i==0:axs[i,j].set_title({'gradcam':'Grad-CAM','gradcam++':'Grad-CAM++','scorecam':'Score-CAM (top channels)'}[kind])
        axs[i,0].set_ylabel(name+'\n'+annotations[name],fontsize=10,rotation=90)
    fig.tight_layout(pad=.6);fig.savefig(dest,dpi=180,bbox_inches='tight');plt.close(fig)

done=[]
for row in chosen.itertuples():
    file=OUT/'figures'/f'{row.source}__{row.image_id}__all_models.png'
    if file.is_file() and file.stat().st_size>30000:
        print('Reusing',file.name);continue
    if time.monotonic()-START>HOURS*3600:break
    raw,processed,x=get_images(row.image_id)
    heatmaps={};notes={};records=[]
    for name in SRC:
        model=wrappers[name]
        gcmap,pred,confidence=gradients_map(model,x,'gradcam')
        gpp,check,_=gradients_map(model,x,'gradcam++')
        req(pred==check,'Prediction changed between Grad-CAM and Grad-CAM++')
        scm,n_channels=scorecam(model,x,pred)
        heatmaps[name]={'gradcam':gcmap,'gradcam++':gpp,'scorecam':scm}
        notes[name]=f'True {CLASSES[row.y]} | Pred {CLASSES[pred]} ({confidence:.2f})'
        np.savez_compressed(OUT/'maps'/f'{row.image_id}_{name}.npz',gradcam=gcmap,gradcampp=gpp,scorecam=scm,
            target_class=np.int32(pred),true_class=np.int32(row.y),confidence=np.float32(confidence))
        records.append({'image_id':row.image_id,'source':row.source,'model':name,'true_label':CLASSES[row.y],
            'predicted_label':CLASSES[pred],'confidence':confidence,'target_for_xai':'PREDICTED_CLASS',
            'target_branch':'simclr' if name not in ['rotation','simclr'] else name,'scorecam_channels':n_channels})
        del gcmap,gpp,scm
    panel(raw,processed,heatmaps,notes,file)
    pd.DataFrame(records).to_csv(OUT/'reports'/f'{row.image_id}_xai_metadata.csv',index=False)
    done.append(row.image_id);torch.cuda.empty_cache();gc.collect()
    print('Created',file.name)
write(OUT/'reports/final_status.json',{'stage':'J13','status':'COMPLETED' if len(list(OUT.glob('figures/*__all_models.png')))==len(chosen) else 'PAUSED',
    'n_selected':len(chosen),'n_finished':len(list((OUT/'figures').glob('*__all_models.png'))),
    'methods':['Grad-CAM','Grad-CAM++','Score-CAM'],'models':SRC,
    'interpretation':'Model-predicted-class attribution for the SimCLR encoder branch in feature fusions, not clinical segmentation.'})
print('XAI files:',len(list((OUT/'figures').glob('*.png'))),'output:',OUT)


Created HAM10000__ISIC_0028428__all_models.png
Created HAM10000__ISIC_0031138__all_models.png
Created HAM10000__ISIC_0027155__all_models.png
Created BCN20000__ISIC_0066252__all_models.png
Created BCN20000__ISIC_0066359__all_models.png
Created BCN20000__ISIC_0064452__all_models.png
XAI files: 6 output: /kaggle/working/j13-lesion-aware-gradcam-plus-scorecam
